# AutoWhisper — Autonomous AI Experiment Loop

Runs on Kaggle T4 GPU. Two modes:
- **Scripted mode**: Pre-defined modification sequence (no API key needed)
- **Agent mode**: Claude API reads `program.md` and proposes modifications

Inspired by [Karpathy's autoresearch](https://github.com/karpathy/autoresearch).

## 1. Setup

In [ ]:
import os
import sys
import time
import subprocess

# Configuration
RUN_TAG = "run_mar17"  # Change per session
MODE = "scripted"  # "scripted" or "agent"
MAX_EXPERIMENTS = 10  # Max experiments per session
SESSION_BUFFER_MIN = 30  # Stop this many minutes before session ends
HF_REPO = "nishantgaurav23/autowhisper-results"  # For checkpoint upload

# Detect Kaggle session time remaining
KAGGLE_SESSION_HOURS = 12  # T4 sessions are 12 hours
SESSION_START = time.time()

def time_remaining_min():
    elapsed = (time.time() - SESSION_START) / 60
    return KAGGLE_SESSION_HOURS * 60 - elapsed

print(f"AutoWhisper | Mode: {MODE} | Tag: {RUN_TAG}")
print(f"Max experiments: {MAX_EXPERIMENTS}")
print(f"Session buffer: {SESSION_BUFFER_MIN} min")

In [ ]:
# Install dependencies (if on Kaggle)
if os.path.exists("/kaggle"):
    !pip install -q jiwer audiomentations peft bitsandbytes

# Clone repo and setup
REPO_DIR = "/kaggle/working/childwhisper" if os.path.exists("/kaggle") else "."
if not os.path.exists(os.path.join(REPO_DIR, "src")):
    !git clone https://github.com/YOUR_USERNAME/childwhisper.git {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Git setup for autowhisper commits
!git config user.email "autowhisper@kaggle.local"
!git config user.name "AutoWhisper Bot"

In [ ]:
# Download competition data
DATA_DIR = "/kaggle/input/pasketti-word-track" if os.path.exists("/kaggle") else "data"
os.environ["AUTOWHISPER_DATA_DIR"] = DATA_DIR
os.environ["AUTOWHISPER_OUTPUT_DIR"] = "/kaggle/working/autowhisper_output"

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {os.environ['AUTOWHISPER_OUTPUT_DIR']}")

## 2. Initialize Run

In [ ]:
from src.autowhisper.runner import init_run, run_experiment, evaluate_and_decide
from src.autowhisper.runner import keep_experiment, revert_experiment, log_result
from src.autowhisper.logger import init_log, get_best_wer, print_summary, plot_progress

RESULTS_DIR = f"results/autowhisper/{RUN_TAG}"
LOG_PATH = f"{RESULTS_DIR}/results.tsv"

os.makedirs(RESULTS_DIR, exist_ok=True)

# Initialize branch and log
branch = init_run(RUN_TAG)
init_log(LOG_PATH)
print(f"Branch: {branch}")
print(f"Log: {LOG_PATH}")

## 3. Run Baseline

In [ ]:
print("Running baseline experiment...")
baseline = run_experiment("src/autowhisper/train.py", time_budget=900)

from src.autowhisper.logger import append_result

baseline_row = {
    "experiment_id": "000",
    "commit_hash": "baseline",
    "val_wer": baseline["val_wer"],
    "peak_vram_mb": baseline["peak_vram_mb"],
    "duration_sec": baseline["duration_sec"],
    "status": "baseline",
    "description": "Baseline configuration",
}
append_result(LOG_PATH, baseline_row)

print(f"Baseline WER: {baseline['val_wer']:.4f}")
print(f"Peak VRAM: {baseline['peak_vram_mb']} MB")
print(f"Duration: {baseline['duration_sec']}s")

## 4. Experiment Loop

### 4a. Scripted Mode

Pre-defined patches applied one-by-one. No API key needed.

In [ ]:
# Pre-defined experiment modifications (scripted mode)
SCRIPTED_EXPERIMENTS = [
    {
        "description": "Lower learning rate to 1e-5",
        "patch": {'learning_rate': 1.0e-5},
    },
    {
        "description": "Increase warmup steps to 500",
        "patch": {'warmup_steps': 500},
    },
    {
        "description": "Higher learning rate 5e-5",
        "patch": {'learning_rate': 5.0e-5},
    },
    {
        "description": "Reduce gradient accumulation to 4",
        "patch": {'gradient_accumulation': 4},
    },
    {
        "description": "Increase beam width to 8",
        "patch": {'num_beams': 8},
    },
    {
        "description": "SpecAugment mask_time_prob 0.08",
        "patch": {'spec_augment_mask_time_prob': 0.08},
    },
    {
        "description": "SpecAugment mask_time_prob 0.02",
        "patch": {'spec_augment_mask_time_prob': 0.02},
    },
    {
        "description": "Increase max_steps to 750",
        "patch": {'max_steps': 750},
    },
    {
        "description": "Batch size 2 with grad_accum 16",
        "patch": {'per_device_batch_size': 2, 'gradient_accumulation': 16},
    },
    {
        "description": "Reduce beam width to 3",
        "patch": {'num_beams': 3},
    },
]


def apply_scripted_patch(patch_dict):
    """Apply a config patch to train.py by rewriting the CONFIG dict."""
    import re
    
    with open("src/autowhisper/train.py") as f:
        content = f.read()
    
    for key, value in patch_dict.items():
        # Match the key in CONFIG dict
        if isinstance(value, float):
            pattern = rf'("{key}":\s*)([\d.e\-+]+)'
            replacement = rf'\g<1>{value}'
        elif isinstance(value, int):
            pattern = rf'("{key}":\s*)(\d+)'
            replacement = rf'\g<1>{value}'
        elif isinstance(value, list):
            pattern = rf'("{key}":\s*)(\[.*?\])'
            replacement = rf'\g<1>{value}'
        else:
            continue
        content = re.sub(pattern, replacement, content)
    
    with open("src/autowhisper/train.py", "w") as f:
        f.write(content)


def restore_train_py():
    """Restore train.py to last committed version."""
    subprocess.run(["git", "checkout", "--", "src/autowhisper/train.py"],
                   capture_output=True)

In [ ]:
if MODE == "scripted":
    for i, exp in enumerate(SCRIPTED_EXPERIMENTS[:MAX_EXPERIMENTS]):
        # Check session time
        if time_remaining_min() < SESSION_BUFFER_MIN:
            print(f"\nSession buffer reached ({SESSION_BUFFER_MIN} min). Stopping.")
            break
        
        exp_id = f"{i + 1:03d}"
        print(f"\n{'=' * 60}")
        print(f"Experiment {exp_id}: {exp['description']}")
        print(f"{'=' * 60}")
        
        # Apply patch
        apply_scripted_patch(exp["patch"])
        
        # Run
        result = run_experiment("src/autowhisper/train.py", time_budget=900)
        best_wer = get_best_wer(LOG_PATH)
        decision = evaluate_and_decide(result, best_wer)
        
        # Decide
        if decision == "keep":
            keep_experiment(exp["description"])
            commit = subprocess.run(
                ["git", "rev-parse", "--short", "HEAD"],
                capture_output=True, text=True
            ).stdout.strip()
            print(f"  -> KEEP (WER: {result['val_wer']:.4f}, was {best_wer:.4f})")
        elif decision == "discard":
            restore_train_py()
            commit = "reverted"
            print(f"  -> DISCARD (WER: {result['val_wer']:.4f}, best: {best_wer:.4f})")
        else:
            restore_train_py()
            commit = "crashed"
            print(f"  -> CRASH")
        
        # Log
        log_result(
            result=result,
            decision=decision,
            description=exp["description"],
            experiment_id=exp_id,
            commit_hash=commit,
            log_path=LOG_PATH,
        )
        
        print(f"  Time remaining: {time_remaining_min():.0f} min")

print("\nExperiment loop complete.")

### 4b. Agent Mode (Optional)

Requires Claude API key as Kaggle secret `ANTHROPIC_API_KEY`.

In [ ]:
if MODE == "agent":
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    api_key = secrets.get_secret("ANTHROPIC_API_KEY")
    
    if not api_key:
        print("ERROR: ANTHROPIC_API_KEY not found in Kaggle secrets.")
        print("Switch to MODE='scripted' or add your API key.")
    else:
        import anthropic
        client = anthropic.Anthropic(api_key=api_key)
        
        # Read program.md for agent instructions
        with open("src/autowhisper/program.md") as f:
            program = f.read()
        
        print("Agent mode initialized. The agent will propose modifications.")
        print("This cell is a placeholder — full agent integration is runtime config.")
        print("See program.md for the agent's instructions.")

## 5. Results & Upload

In [ ]:
# Print summary
print_summary(LOG_PATH)

# Generate progress plot
plot_path = f"{RESULTS_DIR}/progress.png"
plot_progress(LOG_PATH, plot_path)

from IPython.display import Image, display
display(Image(filename=plot_path))

In [ ]:
# Upload best checkpoint and results to HuggingFace Hub
if os.path.exists("/kaggle"):
    from huggingface_hub import HfApi
    api = HfApi()
    
    # Upload results
    api.upload_file(
        path_or_fileobj=LOG_PATH,
        path_in_repo=f"{RUN_TAG}/results.tsv",
        repo_id=HF_REPO,
        repo_type="model",
    )
    api.upload_file(
        path_or_fileobj=plot_path,
        path_in_repo=f"{RUN_TAG}/progress.png",
        repo_id=HF_REPO,
        repo_type="model",
    )
    
    # Upload best train.py (current state after all keeps)
    api.upload_file(
        path_or_fileobj="src/autowhisper/train.py",
        path_in_repo=f"{RUN_TAG}/best_train.py",
        repo_id=HF_REPO,
        repo_type="model",
    )
    
    print(f"Results uploaded to {HF_REPO}/{RUN_TAG}/")
else:
    print("Not on Kaggle — skipping HF Hub upload.")
    print(f"Results saved locally at {RESULTS_DIR}/")